# QA Search Engine using Transformers

# Retrieval & Re-Ranking Pipeline

## Retrieval Stage: Bi-Encoder (Semantic Search)

In the retrieval phase, the goal is to efficiently identify a set of candidate documents relevant to a given query. This can be done using:

- **Lexical Search** (e.g., BM25 / ElasticSearch)
- **Semantic Search** using a **Bi-Encoder**

---

### Lexical Search

Lexical search relies on exact or near-exact keyword matching.

**Limitations:**
- ❌ Does not understand synonyms  
- ❌ Struggles with acronyms/abbreviations  
- ❌ Sensitive to spelling variations  

---

### Semantic Search (Bi-Encoder)

A **Bi-Encoder** encodes queries and documents independently into vector embeddings.

**How it works:**
- Convert query → embedding  
- Convert documents → embeddings (precomputed)  
- Compute similarity (e.g., cosine similarity)  
- Retrieve top-K similar documents  

**Advantages:**
- ✅ Captures semantic meaning  
- ✅ Handles synonyms and paraphrases  
- ✅ Scalable for large datasets  

**Limitations:**
- ❌ No interaction between query & document during encoding  
- ❌ Lower ranking precision compared to Cross-Encoder  

---

## Re-Ranking Stage: Cross-Encoder

The retriever may return some irrelevant candidates. A **Cross-Encoder** improves ranking quality.

### How it works

- Query and document are passed **together** into the model  
- Full attention is applied across both  
- Outputs a **relevance score (0–1)**  

**Advantages:**
- ✅ High accuracy  
- ✅ Captures fine-grained relationships  
- ✅ Better contextual understanding  

**Limitations:**
- ❌ Computationally expensive  
- ❌ Not suitable for large-scale retrieval  

---

## Combined Architecture: Best of Both Worlds

### Step 1: Bi-Encoder Retrieval
- Retrieve top **K candidates** (e.g., 100)

### Step 2: Cross-Encoder Re-Ranking
- Score each *(query, document)* pair  
- Re-rank based on relevance  

---

## Summary

| Stage        | Model Type    | Strengths                         | Limitations                     |
|-------------|--------------|----------------------------------|---------------------------------|
| Retrieval   | Bi-Encoder   | Fast, scalable, semantic search  | Lower precision                 |
| Re-Ranking  | Cross-Encoder| High accuracy, deep interaction  | Computationally expensive       |

---

## Final Outcome

- Efficient retrieval at scale  
- High-quality, precise ranking  

This two-stage pipeline is widely used in:
- RAG (Retrieval-Augmented Generation)
- Search engines
- Enterprise AI systems

### Install Dependencies

In [ ]:
!pip install -U sentence-transformers

### Load Transformer Models, Wikipedia Data and Generate Embeddings

For semantic search, we use `SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')` and retrieve 32 potentially relevant passages that answer the input query.

Next, we use a more powerful CrossEncoder `(cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2'))` that scores the query and all retrieved passages for their relevancy. The cross-encoder further boost the performance.

MS MARCO is a large scale information retrieval corpus that was created based on real user search queries using Bing search engine.

The provided models can be used for semantic search, i.e., given keywords / a search phrase / a question, the model will find passages that are relevant for the search query.

## Load Wikipedia Dataset

In [ ]:
import json
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import gzip
import os
import torch


# As dataset, we use Simple English Wikipedia. Compared to the full English wikipedia, it has only
# about 170k articles. We split these articles into paragraphs and encode them with the bi-encoder

#source - 'http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz'
wikipedia_filepath = 'simplewiki-2020-11-01.jsonl.gz'

passages = []
with gzip.open(wikipedia_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        data = json.loads(line.strip())

        #Add all paragraphs
        #passages.extend(data['paragraphs'])

        #Only add the first paragraph
        passages.append(data['paragraphs'][0])

print("Passages:", len(passages))

## Subset Dataset

In [ ]:
# We subset our data so we only use a subset of wikipedia to run things faster

passages = [passage for passage in passages for x in ['india', 'north pole', 'nlp',
                                                      'natural language processing', 'linguistics',
                                                      'machine learning', 'artificial intelligence',
                                                      'cheetah', 'animal', 'jaguar']
                                                    if x in passage.lower()]

## Look at sample documents

In [ ]:
len(passages)

In [ ]:
passages[4]

## Load Transformer Models

In [ ]:
if not torch.cuda.is_available():
    print("Warning: No GPU found. Please add GPU to your notebook")


# We use the Bi-Encoder to encode all passages, so that we can use it with sematic search
bi_encoder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
# The bi-encoder will retrieve 30 documents.
# We use a cross-encoder, to re-rank the results list to improve the quality
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

## Get Wikipedia Document Embeddings

In [ ]:
# We encode all passages into our vector space. This takes about few seconds (depends on your GPU speed)
corpus_embeddings = bi_encoder.encode(passages)

In [ ]:
passages[4]

In [ ]:
corpus_embeddings[4], corpus_embeddings[4].shape

In [ ]:
corpus_embeddings.shape

## Try Search with a Sample Query

### New Query

In [ ]:
query = "What is the capital of India?"
query

### Get Embedding for New Query

In [ ]:
query_embedding = bi_encoder.encode(query)
query_embedding.shape

### Get Cosine Similarity Score of Document Emebddings compared to New Query Embedding

In [ ]:
cos_scores = util.pytorch_cos_sim(query_embedding, corpus_embeddings)[0]
cos_scores

### Get Most Similar Document ID

In [ ]:
top_results = torch.topk(cos_scores, k=1)
idx = top_results.indices
idx

### Get Most Similar Document

In [ ]:
passages[idx]

## Alternate way of getting most similar document

---



In [ ]:
hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=1)
hits[0]

In [ ]:
hits[0][0]['corpus_id']

## Bi Encoder + ReRanker Cross Encoder Search

### Get top K Similar documents from Bi-encoder and format input data for Reranker Cross-encoder

In [ ]:
# Get top 30 similar documents (hits) to the query
hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=30)
hits = hits[0]
hits

In [ ]:
# Format data for the reranker -> [query, similar_doc] for each of the top_k similar documents
reranker_inp = [[query, passages[hit['corpus_id']]] for hit in hits]
reranker_inp[:10] # look at the first 3 query inputs to the reranker cross encoder model

### Get Reranker score for every similar document

In [ ]:
reranker_scores = cross_encoder.predict(reranker_inp)
reranker_scores[:3] # look at relevance scores from reranker cross encoder

### Add Reranker score back to the hits dictionary

In [ ]:
hits[:3]

In [ ]:
for id, hit in enumerate(hits):
    hit['reranker_score'] = reranker_scores[id]
hits[:3]

In [ ]:
hits

### Show the top similar document to query based on both models

In [ ]:
print("Top Bi-Encoder Retrieval hit: ")
hit = sorted(hits, key=lambda x: x['score'], reverse=True)[0]
print(passages[hit['corpus_id']])

print("Top Reranker Retrieval hit: ")
hit = sorted(hits, key=lambda x: x['reranker_score'], reverse=True)[0]
print(passages[hit['corpus_id']])

## Create a function to return the top similar document based on any query

In [ ]:
def search(query, top_k=30):
  # print the input question
  print("Input question:", query)

  ##### Bi-Encoder: Sematic Search #####
  # Encode the query using the bi-encoder and find potentially relevant passages
  query_embedding = bi_encoder.encode(query)
  hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)
  hits = hits[0]

  ##### Cross-Encoder: Re-Ranking #####
  # Now, score all retrieved passages with the reranker cross encoder
  reranker_inp = [[query, passages[hit['corpus_id']]] for hit in hits]
  reranker_scores = cross_encoder.predict(reranker_inp)

  # Store reranker cross encoder scores back into the hits variable
  for id, hit in enumerate(hits):
      hit['reranker_score'] = reranker_scores[id]

  # Output of top-1 hit from bi-encoder
  print("Top Bi-Encoder Retrieval hit: ")
  hit = sorted(hits, key=lambda x: x['score'], reverse=True)[0]
  print(passages[hit['corpus_id']])

  # Output of top-1 hit from re-ranker
  print("Top Reranker Retrieval hit: ")
  hit = sorted(hits, key=lambda x: x['reranker_score'], reverse=True)[0]
  print(passages[hit['corpus_id']])


## Try out the function

In [ ]:
search(query = "What is the capital of India?")

In [ ]:
search(query = "What is natural language processing?")

In [ ]:
search(query = "What is language?")

In [ ]:
search(query = "What is coldest place on earth?")

In [ ]:
search(query = "What is the animal which can run very fast?")